In [1]:
# Cell 1: clone repo
!git clone https://github.com/nnzhan/Graph-WaveNet.git
%cd Graph-WaveNet

Cloning into 'Graph-WaveNet'...
remote: Enumerating objects: 53, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 53 (delta 12), reused 12 (delta 12), pack-reused 33 (from 1)
Receiving objects: 100% (53/53), 267.08 KiB | 1.30 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/kaggle/working/Graph-WaveNet


In [2]:
# Clone DCRNN để lấy file adj
!git clone https://github.com/liyaguang/DCRNN.git

# Kiểm tra file có ở đó không
!ls DCRNN/data/sensor_graph/

Cloning into 'DCRNN'...
remote: Enumerating objects: 334, done.
remote: Counting objects: 100% (66/66), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 334 (delta 51), reused 34 (delta 34), pack-reused 268 (from 2)
Receiving objects: 100% (334/334), 127.90 MiB | 26.76 MiB/s, done.
Resolving deltas: 100% (153/153), done.
adj_mx_bay.pkl		graph_sensor_ids.txt
adj_mx.pkl		graph_sensor_locations_bay.csv
distances_bay_2017.csv	graph_sensor_locations.csv
distances_la_2012.csv


In [3]:
# Copy file adj vào đúng chỗ Graph-WaveNet cần
!mkdir -p /kaggle/working/Graph-WaveNet/data/sensor_graph

# METR-LA
!cp DCRNN/data/sensor_graph/adj_mx.pkl \
    /kaggle/working/Graph-WaveNet/data/sensor_graph/adj_mx.pkl

# PEMS-BAY
!cp DCRNN/data/sensor_graph/adj_mx_bay.pkl \
    /kaggle/working/Graph-WaveNet/data/sensor_graph/adj_mx_bay.pkl

In [4]:
# Kiểm tra lại
!ls /kaggle/working/Graph-WaveNet/data/sensor_graph/

adj_mx_bay.pkl	adj_mx.pkl


In [5]:
%%writefile model.py
import torch
import torch.nn as nn
import torch.nn.functional as F

# ===========================================================================
# TỔNG HỢP 3 TỐI ƯU ÁP DỤNG:
#   [OPT-1] Low-rank embedding: giảm chiều c từ 10 → EMB_DIM (mặc định 4)
#   [OPT-2] Top-k sparsification: chỉ giữ k cạnh mạnh nhất trên mỗi hàng
#   [OPT-3] Adjacency caching: tính Ã mỗi ADJ_UPDATE_FREQ bước thay vì mỗi forward
#
# CÁCH SỬ DỤNG: tìm comment "# [OPT-x]" để biết đúng vị trí thay đổi.
# Các dòng CODE GỐC được giữ lại dạng comment (# ORIGINAL:) để dễ so sánh.
# ===========================================================================

# ---------------------------------------------------------------------------
# [TUNING] Điều chỉnh 3 hyperparameter này theo nhu cầu:
EMB_DIM = 4          # [OPT-1] chiều embedding (gốc = 10). Thử: 4, 6, 8
TOPK = 10            # [OPT-2] số cạnh giữ lại mỗi node. Thử: 8, 10, 15
ADJ_UPDATE_FREQ = 5  # [OPT-3] tính lại Ã mỗi N bước. Thử: 1 (tắt cache), 5, 10
# ---------------------------------------------------------------------------


class nconv(nn.Module):
    """Nhân tensor đặc trưng x với adjacency matrix A — không thay đổi."""
    def __init__(self):
        super(nconv, self).__init__()

    def forward(self, x, A):
        x = torch.einsum('ncvl,vw->ncwl', (x, A))
        return x.contiguous()


class linear(nn.Module):
    """1×1 Conv để chiếu chiều — không thay đổi."""
    def __init__(self, c_in, c_out):
        super(linear, self).__init__()
        self.mlp = torch.nn.Conv2d(c_in, c_out,
                                   kernel_size=(1, 1),
                                   padding=(0, 0),
                                   stride=(1, 1),
                                   bias=True)

    def forward(self, x):
        return self.mlp(x)


class gcn(nn.Module):
    """Graph Convolution — không thay đổi cấu trúc."""
    def __init__(self, c_in, c_out, dropout, support_len=3, order=2):
        super(gcn, self).__init__()
        self.nconv = nconv()
        c_in = (order * support_len + 1) * c_in
        self.mlp = linear(c_in, c_out)
        self.dropout = dropout
        self.order = order

    def forward(self, x, support):
        out = [x]
        for a in support:
            x1 = self.nconv(x, a)
            out.append(x1)
            for k in range(2, self.order + 1):
                x2 = self.nconv(x1, a)
                out.append(x2)
                x1 = x2
        h = torch.cat(out, dim=1)
        h = self.mlp(h)
        h = F.dropout(h, self.dropout, training=self.training)
        return h


class gwnet(nn.Module):
    def __init__(self, device, num_nodes, dropout=0.3,
                 supports=None, gcn_bool=True, addaptadj=True, aptinit=None,
                 in_dim=2, out_dim=12,
                 residual_channels=32, dilation_channels=32,
                 skip_channels=256, end_channels=512,
                 kernel_size=2, blocks=4, layers=2,
                 # [OPT-1] thêm emb_dim, [OPT-2] thêm topk, [OPT-3] thêm adj_update_freq
                 emb_dim=EMB_DIM, topk=TOPK, adj_update_freq=ADJ_UPDATE_FREQ):
        super(gwnet, self).__init__()

        self.dropout = dropout
        self.blocks = blocks
        self.layers = layers
        self.gcn_bool = gcn_bool
        self.addaptadj = addaptadj
        self.supports = supports

        # [OPT-2] lưu topk để dùng trong forward()
        self.topk = topk

        # [OPT-3] bộ đếm step và cache adjacency
        self._adj_step_counter = 0
        self.adj_update_freq = adj_update_freq
        self._cached_adp = None          # <-- ô nhớ chứa Ã đã tính
        self._cached_new_supports = None # <-- ô nhớ chứa new_supports đầy đủ

        self.filter_convs = nn.ModuleList()
        self.gate_convs = nn.ModuleList()
        self.gconv = nn.ModuleList()

        self.start_conv = nn.Conv2d(in_channels=in_dim,
                                    out_channels=residual_channels,
                                    kernel_size=(1, 1))
        self.residual_convs = nn.ModuleList()
        self.skip_convs = nn.ModuleList()
        self.bn = nn.ModuleList()

        receptive_field = 1
        self.supports_len = 0
        if supports is not None:
            self.supports_len += len(supports)

        # ------------------------------------------------------------------
        # [OPT-1] Thay đổi chiều embedding từ 10 → emb_dim
        # ORIGINAL:
        #   self.nodevec1 = nn.Parameter(torch.randn(num_nodes, 10)...)
        #   self.nodevec2 = nn.Parameter(torch.randn(10, num_nodes)...)
        # ------------------------------------------------------------------
        if gcn_bool and addaptadj:
            if aptinit is None:
                if supports is None:
                    self.supports = []
                # [OPT-1] dùng emb_dim thay vì hardcode 10
                self.nodevec1 = nn.Parameter(
                    torch.randn(num_nodes, emb_dim).to(device),
                    requires_grad=True).to(device)
                self.nodevec2 = nn.Parameter(
                    torch.randn(emb_dim, num_nodes).to(device),
                    requires_grad=True).to(device)
                self.supports_len += 1
            else:
                if supports is None:
                    self.supports = []
                m, p, n = torch.svd(aptinit)
                # [OPT-1] SVD init cũng dùng emb_dim thay vì 10
                # ORIGINAL: initemb1 = torch.mm(m[:, :10], torch.diag(p[:10] ** 0.5))
                initemb1 = torch.mm(m[:, :emb_dim], torch.diag(p[:emb_dim] ** 0.5))
                # ORIGINAL: initemb2 = torch.mm(torch.diag(p[:10] ** 0.5), n[:, :10].t())
                initemb2 = torch.mm(torch.diag(p[:emb_dim] ** 0.5), n[:, :emb_dim].t())
                self.nodevec1 = nn.Parameter(initemb1, requires_grad=True).to(device)
                self.nodevec2 = nn.Parameter(initemb2, requires_grad=True).to(device)
                self.supports_len += 1

        for b in range(blocks):
            additional_scope = kernel_size - 1
            new_dilation = 1
            for i in range(layers):
                self.filter_convs.append(nn.Conv2d(in_channels=residual_channels,
                                                   out_channels=dilation_channels,
                                                   kernel_size=(1, kernel_size),
                                                   dilation=new_dilation))
                self.gate_convs.append(nn.Conv2d(in_channels=residual_channels,
                                 out_channels=dilation_channels,
                                 kernel_size=(1, kernel_size), dilation=new_dilation))
                new_dilation *= 2
                receptive_field += additional_scope
                additional_scope *= 2
                if self.gcn_bool:
                    self.gconv.append(gcn(dilation_channels, residual_channels,
                                          dropout, support_len=self.supports_len))

                self.residual_convs.append(nn.Conv2d(in_channels=dilation_channels,
                                     out_channels=residual_channels,
                                     kernel_size=(1, 1)))

                self.skip_convs.append(nn.Conv2d(in_channels=dilation_channels,
                                                 out_channels=skip_channels,
                                                 kernel_size=(1, 1)))
                self.bn.append(nn.BatchNorm2d(residual_channels))

        self.end_conv_1 = nn.Conv2d(in_channels=skip_channels,
                                    out_channels=end_channels,
                                    kernel_size=(1, 1), bias=True)
        self.end_conv_2 = nn.Conv2d(in_channels=end_channels,
                                    out_channels=out_dim,
                                    kernel_size=(1, 1), bias=True)
        self.receptive_field = receptive_field

    # ------------------------------------------------------------------
    # [OPT-2] Helper: áp dụng top-k sparsification lên ma trận adp [N×N]
    # ------------------------------------------------------------------
    def _topk_sparse(self, adp):
        """Chỉ giữ self.topk giá trị lớn nhất trên mỗi hàng, đặt phần còn lại = 0."""
        k = min(self.topk, adp.size(1))         # đề phòng k > N
        topk_vals, topk_idx = torch.topk(adp, k, dim=1)
        mask = torch.zeros_like(adp)
        mask.scatter_(1, topk_idx, 1.0)
        return adp * mask                        # zero-out các cạnh yếu

    # ------------------------------------------------------------------
    # [OPT-3] Helper: tính Ã đầy đủ (bao gồm top-k) và lưu vào cache
    # ------------------------------------------------------------------
    def _compute_and_cache_adj(self):
        """Tính adp mới → top-k → ghép với self.supports → lưu cache."""
        # Bước 1: tính full N×N (vẫn cần, nhưng chỉ chạy mỗi adj_update_freq bước)
        adp_full = F.softmax(F.relu(torch.mm(self.nodevec1, self.nodevec2)), dim=1)
        # [OPT-2] Bước 2: sparsify
        adp_sparse = self._topk_sparse(adp_full)
    
        # THÊM .detach() — tách khỏi computation graph trước khi lưu cache
        adp_cached = adp_sparse.detach()

        # Bước 3: ghép với fixed supports
        self._cached_adp = adp_cached
        self._cached_new_supports = self.supports + [adp_cached]

    def forward(self, input):
        in_len = input.size(3)
        if in_len < self.receptive_field:
            x = nn.functional.pad(input, (self.receptive_field - in_len, 0, 0, 0))
        else:
            x = input

        x = self.start_conv(x)
        skip = 0

        # ------------------------------------------------------------------
        # [OPT-2] + [OPT-3] Tính Adaptive Adjacency Matrix
        #
        # CODE GỐC (tính mỗi forward, không sparse):
        #   adp = F.softmax(F.relu(torch.mm(self.nodevec1, self.nodevec2)), dim=1)
        #   new_supports = self.supports + [adp]
        #
        # CODE MỚI: cache + top-k sparse
        # ------------------------------------------------------------------
        new_supports = None
        if self.gcn_bool and self.addaptadj and self.supports is not None:
            if self.training:
                should_update = (self._cached_new_supports is None or
                                 self._adj_step_counter % self.adj_update_freq == 0)
    
                if should_update:
                    # Bước CẬP NHẬT: tính adp mới, có gradient → train nodevec
                    adp_full = F.softmax(F.relu(torch.mm(self.nodevec1, self.nodevec2)), dim=1)
                    adp_live = self._topk_sparse(adp_full)     # có gradient
                    self._compute_and_cache_adj()               # lưu detached vào cache
                    new_supports = self.supports + [adp_live]  # dùng bản CÓ gradient
                else:
                    # Bước DÙNG CACHE: detached, không train nodevec bước này
                    new_supports = self._cached_new_supports
    
                self._adj_step_counter += 1
            else:
                # Eval: luôn tính mới, detach vì không cần backward
                adp_full = F.softmax(F.relu(torch.mm(self.nodevec1, self.nodevec2)), dim=1)
                adp_sparse = self._topk_sparse(adp_full)
                new_supports = self.supports + [adp_sparse.detach()]

        # ------------------------------------------------------------------
        # WaveNet layers — không thay đổi cấu trúc vòng lặp
        # ------------------------------------------------------------------
        for i in range(self.blocks * self.layers):

            residual = x

            # TCN-a: tanh branch
            filter = self.filter_convs[i](residual)
            filter = torch.tanh(filter)

            # TCN-b: sigmoid (gate) branch
            gate = self.gate_convs[i](residual)
            gate = torch.sigmoid(gate)

            # Gated output
            x = filter * gate

            # Skip connection
            s = x
            s = self.skip_convs[i](s)
            try:
                skip = skip[:, :, :, -s.size(3):]
            except Exception:
                skip = 0
            skip = s + skip

            # GCN với new_supports (đã bao gồm adp sparse từ cache)
            if self.gcn_bool and self.supports is not None:
                if self.addaptadj:
                    x = self.gconv[i](x, new_supports)
                else:
                    x = self.gconv[i](x, self.supports)
            else:
                x = self.residual_convs[i](x)

            # Residual connection
            x = x + residual[:, :, :, -x.size(3):]
            x = self.bn[i](x)

        x = F.relu(skip)
        x = F.relu(self.end_conv_1(x))
        x = self.end_conv_2(x)
        return x

Overwriting model.py


In [6]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/annnnguyen/metr-la-dataset/adj_METR-LA.pkl
/kaggle/input/datasets/annnnguyen/metr-la-dataset/METR-LA.h5
/kaggle/input/datasets/nhihhunhth/graohwavenet/_epoch_53_2.79.pth
/kaggle/input/datasets/nhihhunhth/graohwavenet/_epoch_80_1.64.pth
/kaggle/input/datasets/nhihhunhth/graohwavenet/_epoch_76_1.66.pth
/kaggle/input/datasets/nhihhunhth/graohwavenet/_epoch_73_1.63.pth
/kaggle/input/datasets/nhihhunhth/graohwavenet/_epoch_89_1.64.pth
/kaggle/input/datasets/nhihhunhth/graohwavenet/model_fixed.py
/kaggle/input/datasets/nhihhunhth/graohwavenet/_epoch_74_2.81.pth
/kaggle/input/datasets/nhihhunhth/graohwavenet/metr_epoch_96_1.62.pth
/kaggle/input/datasets/nhihhunhth/graohwavenet/metr_epoch_94_1.61.pth
/kaggle/input/datasets/nhihhunhth/graohwavenet/_epoch_88_2.78.pth
/kaggle/input/datasets/scchuy/pemsbay/adj_mx_bay.pkl
/kaggle/input/datasets/scchuy/pemsbay/pems-bay.h5
/kaggle/input/datasets/scchuy/pemsbay/pems-bay-meta.h5


In [7]:
!rm -rf data/METR-LA data/PEMS-BAY

!python generate_training_data.py \
    --output_dir=data/METR-LA \
    --traffic_df_filename=/kaggle/input/datasets/annnnguyen/metr-la-dataset/METR-LA.h5

!python generate_training_data.py \
    --output_dir=data/PEMS-BAY \
    --traffic_df_filename=/kaggle/input/datasets/scchuy/pemsbay/pems-bay.h5

x shape:  (34249, 12, 207, 2) , y shape:  (34249, 12, 207, 2)
train x:  (23974, 12, 207, 2) y: (23974, 12, 207, 2)
val x:  (3425, 12, 207, 2) y: (3425, 12, 207, 2)
test x:  (6850, 12, 207, 2) y: (6850, 12, 207, 2)
x shape:  (52093, 12, 325, 2) , y shape:  (52093, 12, 325, 2)
train x:  (36465, 12, 325, 2) y: (36465, 12, 325, 2)
val x:  (5209, 12, 325, 2) y: (5209, 12, 325, 2)
test x:  (10419, 12, 325, 2) y: (10419, 12, 325, 2)


In [8]:
# Tạo thư mục lưu checkpoint trước
!mkdir -p garage

In [9]:
!python train.py \
    --device cuda:0 \
    --data data/PEMS-BAY \
    --adjdata data/sensor_graph/adj_mx.pkl \
    --adjtype doubletransition \
    --gcn_bool \
    --addaptadj \
    --num_nodes 325 \
    --save garage/ \
    --expid 2

Traceback (most recent call last):
  File "/kaggle/working/Graph-WaveNet/train.py", line 173, in <module>
    main()
  File "/kaggle/working/Graph-WaveNet/train.py", line 46, in main
    supports = [torch.tensor(i).to(device) for i in adj_mx]
                ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py", line 417, in _lazy_init
    raise AssertionError("Torch not compiled with CUDA enabled")
AssertionError: Torch not compiled with CUDA enabled


In [10]:
# !python test.py \
#     --device cuda:0 \
#     --data data/METR-LA \
#     --adjdata data/sensor_graph/adj_mx.pkl \
#     --adjtype doubletransition \
#     --gcn_bool \
#     --addaptadj \
#     --num_nodes 207 \
#     --checkpoint /kaggle/input/datasets/nhihhunhth/graohwavenet/_epoch_88_2.78.pth